# 10 — Hiperparâmetros, inicialização e treinamento

**EP · Data Science Knowledge Base | Deep Learning | Notebook 10/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/10_hiperparametros_inicializacao.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_hiperparametros_inicializacao.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Diferenciar parâmetros treináveis de hiperparâmetros e estudar escala das inicializações.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

Arquitetura, tamanho do batch, taxa de aprendizado, inicialização e número de épocas são hiperparâmetros. Inicialização inadequada pode saturar ativações ou provocar variâncias instáveis. Manter orçamento de busca e validação explícitos.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `Var(W_He)≈2/fan_in`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import numpy as np
rng=np.random.default_rng(2)
fan_in=256
w=rng.normal(0,np.sqrt(2/fan_in),size=(fan_in,64))
print('Desvio He esperado=',round(np.sqrt(2/fan_in),4),'observado=',round(w.std(),4))
assert w.shape==(256,64)

Desvio He esperado= 0.0884 observado= 0.0884


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    keras.utils.set_random_seed(42)
    rede=keras.Sequential([keras.Input(shape=(16,)),layers.Dense(32,activation='relu',kernel_initializer='he_normal'),layers.Dense(1)])
    rede.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss='mse');rede.summary()

Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.


## 5. Exercício de fixação

**Questão:** Uma camada com 128 entradas e 32 unidades Dense tem quantos parâmetros?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** 128×32 pesos + 32 vieses = 4.128 parâmetros.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** Arquitetura, tamanho do batch, taxa de aprendizado, inicialização e número de épocas são hiperparâmetros.

**Expressão:** `Var(W_He)≈2/fan_in`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Seleção de hiperparâmetros dentro do treino, teste intocado

A busca de hiperparâmetros deve ocorrer somente dentro dos dados de treinamento com validação cruzada reprodutível.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score

# Dados reproduzíveis
X, y = make_classification(
    n_samples=180,
    n_features=7,
    n_informative=5,
    random_state=9
)

Xtr, Xte, ytr, yte = train_test_split(
    X, y,
    stratify=y,
    test_size=0.25,
    random_state=9
)

# Pipeline sem vazamento da padronização
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
    solver="lbfgs",
    max_iter=1000,
    max_fun=15000,
    random_state=9
))
])

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=9
)

search = GridSearchCV(
    pipe,
    {
        "mlp__hidden_layer_sizes": [(8,), (12,)],
        "mlp__alpha": [0.001, 0.1]
    },
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=1
)

search.fit(Xtr, ytr)

mlp = search.best_estimator_.named_steps["mlp"]

print("Melhor configuração:", search.best_params_)
print("Balanced Accuracy CV:", round(search.best_score_, 3))
print(
    "Balanced Accuracy teste:",
    round(balanced_accuracy_score(yte, search.predict(Xte)), 3)
)
print("Iterações do melhor modelo:", mlp.n_iter_)
print("Perda final:", round(mlp.loss_, 4))

Melhor configuração: {'mlp__alpha': 0.1, 'mlp__hidden_layer_sizes': (8,)}
Balanced Accuracy CV: 0.845
Balanced Accuracy teste: 0.733
Iterações do melhor modelo: 239
Perda final: 0.1182


### Exportação de resultados (local)

Execute esta célula após o experimento acima. Os arquivos são salvos em `outputs/` e não entram automaticamente no Git.

In [4]:
# Exportação da validação cruzada; NÃO otimizar a partir do conjunto de teste.
from pathlib import Path
import pandas as pd

out=Path('outputs'); out.mkdir(exist_ok=True)
colunas=['params','mean_test_score','std_test_score','rank_test_score']
resultados=pd.DataFrame(search.cv_results_)[colunas].copy()
resultados['params']=resultados['params'].map(str)
caminho=out/'10_resultados_grid_search.csv'
resultados.to_csv(caminho,index=False)
print('Exportado:',caminho)


Exportado: outputs/10_resultados_grid_search.csv


### Interpretação, limites e verificação

A métrica no conjunto de teste só deve ser calculada após escolher a configuração, sem repetidamente ajustá-la ao teste.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.